In [ ]:
import torch
inputs=torch.tensor([
    [0.43,0.15,0.8,0.55,0.87,0.66], # your
    [0.57,0.85,0.64,0.22,0.58,0.23], # journey
    [0.77,0.25,0.10,0.05,0.80,0.55], # starts
])

In [ ]:
class multi_head_attention(torch.nn.Module):
    def __init__(self, d_in,d_out,context_length,drop_out,num_heads,qkv_bias=False):
        super().__init__()
        
        assert (d_out%num_heads==0), \
            "d_out must be divisible by number of heads"
            
        self.d_out=d_out
        self.d_in=d_in
        self.head_dim=d_out//num_heads
        self.num_heads=num_heads
        self.out_proj=torch.nn.Linear(d_out, d_out)
        
        self.W_query=torch.nn.Linear(d_in,d_out, bias=qkv_bias)
        self.W_key=torch.nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value=torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        
        self.dropout=torch.nn.Dropout(drop_out)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length),diagonal=1) )
        
    def forward(self,x):
        b,num_tokens,d_in=x.shape
        keys=self.W_key(x)
        values=self.W_value(x)
        query=self.W_query(x)
        
        # Now we split the b,num_tokens,d_out to b, num_tokens, num_heads, head_dim
        
        keys=keys.view(b,num_tokens,self.num_heads,self.head_dim)
        values=values.view(b, num_tokens, self.num_heads, self.head_dim)
        query=query.view(b, num_tokens, self.num_heads, self.head_dim)
        
        # Now group the matrices by head
        
        keys=keys.transpose(1,2) # b, num_heads, num_tokens, head_dim
        values=values.transpose(1,2)
        query=query.transpose(1,2)
        
        attention_scores = query @ keys.transpose(2,3)
        mask_bool=self.mask.bool()[:num_tokens,:num_tokens]
        attention_scores.masked_fill_(mask_bool, float('-inf'))
        
        attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5, dim=-1)
        attention_weights=self.dropout(attention_weights) # shape is b, num_head,num_tokens,num_tokens
        context_vector =(attention_weights @ values).transpose(1,2)
        context_vector=context_vector.contiguous().view(b,num_tokens,self.d_out)
        context_vector= self.out_proj(context_vector)
        return context_vector
        

In [ ]:
torch.manual_seed(123)
inputs=torch.tensor([
    [0.43,0.15,0.8,0.55,0.87,0.66], # your
    [0.57,0.85,0.64,0.22,0.58,0.23], # journey
    [0.77,0.25,0.10,0.05,0.80,0.55], # starts
])

batch=torch.stack((inputs,inputs),dim=0)
print(batch.shape)
b,context_len,d_in=batch.shape
d_out=6
mha=multi_head_attention(d_in,d_out,context_len,0.1,2)
context_vec=mha(batch)


In [ ]:
context_vec